# 数据获取与初步探索性数据分析

本notebook对应S1初期阶段，主要任务：
- 数据获取与加载
- 初步探索性数据分析
- 数据质量评估


In [ ]:
# 设置文件路径
file_path = r"D:\railway_risk_analysis\Rail_Equipment_Accident_Incident_Data__Form_54__20251013.csv"

# 加载FRA REAIR数据
print("正在加载FRA REAIR数据...")
df = pd.read_csv(file_path, low_memory=False)
print(f"数据加载完成！数据形状: {df.shape}")
print(f"列数: {df.shape[1]}, 行数: {df.shape[0]}")


In [ ]:
# 显示前5行数据
print("=" * 80)
print("前5行数据预览:")
print("=" * 80)
print(df.head())


In [ ]:
# 显示所有列名
print("=" * 80)
print("所有列名:")
print("=" * 80)
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")
print(f"\n总共有 {len(df.columns)} 列")


In [ ]:
# 缺失值统计
print("=" * 80)
print("缺失值统计:")
print("=" * 80)
missing_stats = df.isnull().sum()
missing_percentage = (missing_stats / len(df)) * 100

missing_df = pd.DataFrame({
    '列名': missing_stats.index,
    '缺失值数量': missing_stats.values,
    '缺失值百分比': missing_percentage.values
}).sort_values('缺失值数量', ascending=False)

print(missing_df.to_string(index=False))
print(f"\n总缺失值: {missing_stats.sum()}")
print(f"完全缺失的列数: {(missing_stats == len(df)).sum()}")
print(f"无缺失值的列数: {(missing_stats == 0).sum()}")


In [ ]:
# 数据类型分析
print("=" * 80)
print("数据类型分析:")
print("=" * 80)
print(df.dtypes.value_counts())
print(f"\n数值型列数: {len(df.select_dtypes(include=[np.number]).columns)}")
print(f"文本型列数: {len(df.select_dtypes(include=['object']).columns)}")
print(f"日期型列数: {len(df.select_dtypes(include=['datetime64']).columns)}")


In [ ]:
# 数值列的基本统计摘要
print("=" * 80)
print("数值列的基本统计摘要:")
print("=" * 80)

# 获取数值列
numeric_columns = df.select_dtypes(include=[np.number]).columns
print(f"数值列数量: {len(numeric_columns)}")
print(f"数值列名称: {list(numeric_columns)}")

if len(numeric_columns) > 0:
    print("\n基本统计信息:")
    print(df[numeric_columns].describe())
else:
    print("没有发现数值型列")


In [ ]:
# 数据质量评估
print("=" * 80)
print("数据质量评估:")
print("=" * 80)

# 重复行检查
duplicate_rows = df.duplicated().sum()
print(f"重复行数量: {duplicate_rows}")

# 内存使用情况
memory_usage = df.memory_usage(deep=True).sum() / 1024**2  # 转换为MB
print(f"内存使用: {memory_usage:.2f} MB")

# 检查是否有完全空白的行或列
empty_rows = df.isnull().all(axis=1).sum()
empty_cols = df.isnull().all(axis=0).sum()
print(f"完全空白的行数: {empty_rows}")
print(f"完全空白的列数: {empty_cols}")

# 数据时间范围（如果有日期列）
date_columns = df.select_dtypes(include=['datetime64']).columns
if len(date_columns) > 0:
    print(f"\n日期列: {list(date_columns)}")
    for col in date_columns:
        print(f"{col} 范围: {df[col].min()} 到 {df[col].max()}")
else:
    print("\n未发现日期型列")


In [ ]:
# 保存数据到processed目录
print("=" * 80)
print("保存数据到processed目录:")
print("=" * 80)

# 确保processed目录存在
import os
processed_dir = "../data/processed"
os.makedirs(processed_dir, exist_ok=True)

# 保存原始数据
output_file = os.path.join(processed_dir, "fra_reair_data_loaded.csv")
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"数据已保存到: {output_file}")

# 保存数据摘要
summary_file = os.path.join(processed_dir, "data_summary.txt")
with open(summary_file, 'w', encoding='utf-8') as f:
    f.write("FRA REAIR 数据摘要\n")
    f.write("=" * 50 + "\n")
    f.write(f"数据形状: {df.shape}\n")
    f.write(f"列数: {df.shape[1]}, 行数: {df.shape[0]}\n")
    f.write(f"缺失值总数: {df.isnull().sum().sum()}\n")
    f.write(f"重复行数: {df.duplicated().sum()}\n")
    f.write(f"内存使用: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB\n")

print(f"数据摘要已保存到: {summary_file}")
print("数据加载和分析完成！")


In [ ]:
# 日期列转换和特征工程
print("=" * 80)
print("日期列转换和特征工程:")
print("=" * 80)

# 检查日期相关的列
date_related_cols = [col for col in df.columns if any(keyword in col.lower() for keyword in ['date', 'time', 'year', 'month', 'day'])]
print(f"发现日期相关列: {date_related_cols}")

# 显示日期相关列的数据类型和样本
print("\n日期相关列信息:")
for col in date_related_cols[:10]:  # 只显示前10个
    print(f"{col}: {df[col].dtype}")
    print(f"  样本值: {df[col].dropna().head(3).tolist()}")
    print(f"  缺失值: {df[col].isnull().sum()}")
    print()


In [ ]:
# 使用日期转换函数
from src.data_loader import convert_fra_date_columns, create_date_features

print("开始转换日期列...")

# 转换日期列
df_with_dates = convert_fra_date_columns(
    df, 
    year_col='Accident Year', 
    month_col='Accident Month',
    day_col='Day' if 'Day' in df.columns else None,
    date_col='Date' if 'Date' in df.columns else None
)

print(f"转换后数据形状: {df_with_dates.shape}")
print(f"新增列数: {df_with_dates.shape[1] - df.shape[1]}")

# 检查新创建的日期列
new_date_cols = [col for col in df_with_dates.columns if col not in df.columns]
print(f"\n新创建的日期相关列: {new_date_cols}")

# 显示accident_date列的信息
if 'accident_date' in df_with_dates.columns:
    print(f"\naccident_date列信息:")
    print(f"  数据类型: {df_with_dates['accident_date'].dtype}")
    print(f"  非空值数量: {df_with_dates['accident_date'].notna().sum()}")
    print(f"  缺失值数量: {df_with_dates['accident_date'].isnull().sum()}")
    print(f"  日期范围: {df_with_dates['accident_date'].min()} 到 {df_with_dates['accident_date'].max()}")
    print(f"  样本值:")
    print(df_with_dates['accident_date'].dropna().head())


In [ ]:
# 创建额外的日期特征
print("创建额外的日期特征...")

# 从accident_date创建更多特征
df_with_date_features = create_date_features(df_with_dates, 'accident_date')

print(f"添加日期特征后数据形状: {df_with_date_features.shape}")

# 显示新创建的日期特征
new_features = [col for col in df_with_date_features.columns if col.startswith('accident_date_')]
print(f"\n新创建的日期特征: {new_features}")

# 显示日期特征的统计信息
if new_features:
    print("\n日期特征统计:")
    for feature in new_features[:5]:  # 显示前5个特征
        if df_with_date_features[feature].dtype == 'object':
            print(f"{feature}: {df_with_date_features[feature].value_counts().head()}")
        else:
            print(f"{feature}: 范围 {df_with_date_features[feature].min()} - {df_with_date_features[feature].max()}")


In [ ]:
# 保存处理后的数据
print("=" * 80)
print("保存处理后的数据:")
print("=" * 80)

# 保存带有日期特征的数据
output_file_with_dates = os.path.join(processed_dir, "fra_reair_data_with_dates.csv")
df_with_date_features.to_csv(output_file_with_dates, index=False, encoding='utf-8-sig')
print(f"[成功] 带日期特征的数据已保存到: {output_file_with_dates}")

# 保存日期转换摘要
date_summary_file = os.path.join(processed_dir, "date_conversion_summary.txt")
with open(date_summary_file, 'w', encoding='utf-8') as f:
    f.write("FRA REAIR 日期转换摘要\n")
    f.write("=" * 50 + "\n")
    f.write(f"原始数据形状: {df.shape}\n")
    f.write(f"转换后数据形状: {df_with_date_features.shape}\n")
    f.write(f"新增列数: {df_with_date_features.shape[1] - df.shape[1]}\n")
    
    if 'accident_date' in df_with_date_features.columns:
        f.write(f"\naccident_date列信息:\n")
        f.write(f"  非空值数量: {df_with_date_features['accident_date'].notna().sum()}\n")
        f.write(f"  缺失值数量: {df_with_date_features['accident_date'].isnull().sum()}\n")
        f.write(f"  日期范围: {df_with_date_features['accident_date'].min()} 到 {df_with_date_features['accident_date'].max()}\n")
    
    f.write(f"\n新创建的日期特征: {new_features}\n")

print(f"[成功] 日期转换摘要已保存到: {date_summary_file}")
print("日期转换和特征工程完成！")


In [ ]:
# 缺失值处理策略分析
print("=" * 80)
print("缺失值处理策略分析:")
print("=" * 80)

from src.data_loader import suggest_missing_value_strategies, handle_fra_missing_values

# 获取缺失值处理策略建议
strategies = suggest_missing_value_strategies(df)

print(f"高缺失值列 (建议删除): {len(strategies['high_missing_columns'])}个")
if strategies['high_missing_columns']:
    print("  列名:", strategies['high_missing_columns'][:5])  # 显示前5个

print(f"\n数值型列处理建议: {len(strategies['numeric_columns'])}个")
for item in strategies['numeric_columns'][:3]:  # 显示前3个
    print(f"  {item['column']}: {item['description']}")

print(f"\n文本型列处理建议: {len(strategies['text_columns'])}个")
for item in strategies['text_columns'][:3]:  # 显示前3个
    print(f"  {item['column']}: {item['description']}")

print(f"\n特殊列处理建议: {len(strategies['special_columns'])}个")
for item in strategies['special_columns'][:3]:  # 显示前3个
    print(f"  {item['column']} ({item['category']}): 缺失率 {item['missing_ratio']:.2%}")

print(f"\n总体建议:")
for i, rec in enumerate(strategies['recommendations'][:5], 1):
    print(f"  {i}. {rec}")


In [ ]:
# 执行缺失值处理
print("=" * 80)
print("执行缺失值处理:")
print("=" * 80)

# 执行智能缺失值处理
df_cleaned = handle_fra_missing_values(
    df, 
    high_missing_threshold=0.95,  # 删除缺失率>95%的列
    numeric_missing_threshold=0.10,  # 数值列缺失率<10%时填充
    text_missing_threshold=0.20,  # 文本列缺失率<20%时填充
    placeholder='UNKNOWN'  # 文本列填充占位符
)

print(f"处理前数据形状: {df.shape}")
print(f"处理后数据形状: {df_cleaned.shape}")
print(f"删除列数: {df.shape[1] - df_cleaned.shape[1]}")

# 检查处理后的缺失值情况
final_missing = df_cleaned.isnull().sum()
high_missing_remaining = final_missing[final_missing > len(df_cleaned) * 0.5]
print(f"\n剩余高缺失值列: {len(high_missing_remaining)}个")
if len(high_missing_remaining) > 0:
    print("  列名:", high_missing_remaining.index.tolist()[:5])

# 显示处理效果
print(f"\n处理效果:")
print(f"  总缺失值减少: {df.isnull().sum().sum() - df_cleaned.isnull().sum().sum()}")
print(f"  完全无缺失值的列: {(final_missing == 0).sum()}")
print(f"  缺失率<5%的列: {(final_missing / len(df_cleaned) < 0.05).sum()}")


In [ ]:
# 保存清理后的数据
print("=" * 80)
print("保存清理后的数据:")
print("=" * 80)

# 保存清理后的数据
cleaned_output_file = os.path.join(processed_dir, "fra_reair_data_cleaned.csv")
df_cleaned.to_csv(cleaned_output_file, index=False, encoding='utf-8-sig')
print(f"[成功] 清理后的数据已保存到: {cleaned_output_file}")

# 保存缺失值处理摘要
missing_summary_file = os.path.join(processed_dir, "missing_values_handling_summary.txt")
with open(missing_summary_file, 'w', encoding='utf-8') as f:
    f.write("FRA REAIR 缺失值处理摘要\n")
    f.write("=" * 50 + "\n")
    f.write(f"原始数据形状: {df.shape}\n")
    f.write(f"清理后数据形状: {df_cleaned.shape}\n")
    f.write(f"删除列数: {df.shape[1] - df_cleaned.shape[1]}\n")
    f.write(f"原始总缺失值: {df.isnull().sum().sum()}\n")
    f.write(f"清理后总缺失值: {df_cleaned.isnull().sum().sum()}\n")
    f.write(f"缺失值减少: {df.isnull().sum().sum() - df_cleaned.isnull().sum().sum()}\n")
    f.write(f"完全无缺失值的列: {(df_cleaned.isnull().sum() == 0).sum()}\n")
    f.write(f"缺失率<5%的列: {(df_cleaned.isnull().sum() / len(df_cleaned) < 0.05).sum()}\n")
    
    f.write(f"\n高缺失值列 (已删除):\n")
    for col in strategies['high_missing_columns']:
        f.write(f"  - {col}\n")
    
    f.write(f"\n处理策略:\n")
    for rec in strategies['recommendations']:
        f.write(f"  - {rec}\n")

print(f"[成功] 缺失值处理摘要已保存到: {missing_summary_file}")
print("缺失值处理完成！")


In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load_fra_reair_data
